## Data Filteration of Processed Raw Data

In [1]:
import os
import pandas as pd
import numpy as np
import sqlite3
import mysql.connector as sql
import warnings

### Raw Datasets

In [2]:
raw_constituency_summary = pd.read_csv('ProcessedData/raw_constituency_summary.csv')
raw_election_results_1951_2019 = pd.read_csv('ProcessedData/raw_election_results_1951_2019.csv')
raw_election_results_2024 = pd.read_csv('ProcessedData/raw_election_results_2024.csv')
raw_literacy_1951_2011 = pd.read_csv('ProcessedData/raw_literacy_1951_2011.csv')
raw_loksabha_1962_2019 = pd.read_csv('ProcessedData/raw_loksabha_1962_2019.csv')
raw_parliament_1951_2014 = pd.read_csv('ProcessedData/raw_parliament_1951_2014.csv')
raw_ref_party_master = pd.read_csv('ProcessedData/raw_ref_party_master.csv')
raw_state_gdp_share = pd.read_csv('ProcessedData/raw_state_gdp_share.csv')
raw_state_sdp = pd.read_csv('ProcessedData/raw_state_sdp.csv')

### Reference Datasets for State Name, Birth of State and Capital

In [3]:
state_crosswalk = pd.read_csv("ReferenceData/state_crosswalk.csv")
state_name_audit = pd.read_csv("ReferenceData/state_name_audit.csv")
state_name_variants = pd.read_csv("ReferenceData/state_name_variants.csv")

---

### Table 1: raw_election_results_1951_2019

In [19]:
# ─── Step 1: Check actual column names ────────────────────────────────────
raw_election_results_1951_2019.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 89840 entries, 0 to 89839
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   89829 non-null  float64
 1   state                89840 non-null  object 
 2   constitution         89840 non-null  object 
 3   election_year        89840 non-null  object 
 4   candidate            89840 non-null  object 
 5   gender               85319 non-null  object 
 6   party                89824 non-null  object 
 7   age                  29470 non-null  float64
 8   category             33882 non-null  object 
 9   votes_received       89834 non-null  float64
 10  votes_received_perc  89811 non-null  object 
 11  num_seats            89840 non-null  int64  
 12  state_pc_year        89840 non-null  object 
 13  rank                 89840 non-null  int64  
 14  result               89840 non-null  object 
dtypes: float64(3), int64(2), object(10)


In [20]:
raw_election_results_1951_2019.head()

,id,state,constitution,election_year,candidate,gender,party,age,category,votes_received,votes_received_perc,num_seats,state_pc_year,rank,result
0,1505.0,Hyderabad,ADILABAD,1951 (1st LOK SABHA),C. MADHAV REDDY,NaN,SP,NaN,NaN,90995.0,57.99%,1,Hyderabad-ADILABAD-1951 (1st LOK SABHA),1,winner
1,1506.0,Hyderabad,ADILABAD,1951 (1st LOK SABHA),J. V. NARSINGRAO,NaN,INC,NaN,NaN,65912.0,42.01%,1,Hyderabad-ADILABAD-1951 (1st LOK SABHA),2,1st runner up
2,1032.0,Uttar Pradesh,AGRA DISTRICT (EAST),1951 (1st LOK SABHA),RAGHUBIR SINGH,NaN,INC,NaN,NaN,70154.0,44.85%,1,Uttar Pradesh-AGRA DISTRICT (EAST)-1951 (1st L...,1,winner
3,1033.0,Uttar Pradesh,AGRA DISTRICT (EAST),1951 (1st LOK SABHA),HIRDAY NAND KUNZRU,NaN,IND,NaN,NaN,47642.0,30.46%,1,Uttar Pradesh-AGRA DISTRICT (EAST)-1951 (1st L...,2,1st runner up
4,1034.0,Uttar Pradesh,AGRA DISTRICT (EAST),1951 (1st LOK SABHA),INDER JIT,NaN,IND,NaN,NaN,16577.0,10.60%,1,Uttar Pradesh-AGRA DISTRICT (EAST)-1951 (1st L...,3,2nd runner up


In [27]:
cols_to_numeric = ['id', 'age', 'votes_received']
for col in cols_to_numeric:
    raw_election_results_1951_2019[col] = pd.to_numeric(
        raw_election_results_1951_2019[col], errors='coerce'
    )
    print(f" Converted {col}")


raw_election_results_1951_2019['election_year_clean'] = pd.to_numeric(
    raw_election_results_1951_2019['election_year'], errors='coerce'
)

 Converted id
 Converted age
 Converted votes_received


In [ ]:
# ─── 1. Fix election_year_clean  ───────────────
# Extract year from "1951 (1st LOK SABHA)" format
if raw_election_results_1951_2019['election_year_clean'].isna().all():
    raw_election_results_1951_2019['election_year_clean'] = (
        raw_election_results_1951_2019['election_year']
        .str.split()
        .str[0]  # First word is always the year
        .astype('Int64')
    )

raw_election_results_1951_2019['vote_share_pct'] = (
    raw_election_results_1951_2019['votes_received_perc']
    .astype(str)
    .str.replace('%', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)

# ─── 2. Verify vote_share_pct is working correctly ───────────────────────
print("Vote share stats (should look good):")
print(raw_election_results_1951_2019['vote_share_pct'].describe())

print("\\nYear extraction verification:")
print(raw_election_results_1951_2019['election_year_clean'].value_counts().sort_index().head(10))
print("\\nSample year mapping:")
print(raw_election_results_1951_2019[['election_year', 'election_year_clean']].head(10))

# ________________ state Mapping__________________________
state_mapping = dict(zip(state_name_variants['variant'], state_name_variants['canonical_name']))
raw_election_results_1951_2019['state_raw'] = raw_election_results_1951_2019['state']
raw_election_results_1951_2019['state_canonical'] = raw_election_results_1951_2019['state'].map(state_mapping).fillna(raw_election_results_1951_2019['state'])

# ─── 3. State mapping verification ───────────────────────────────────────
print("\\nState mapping verification:")
unmapped_states = raw_election_results_1951_2019[
    raw_election_results_1951_2019['state_raw'] != raw_election_results_1951_2019['state_canonical']
]
print(f"Successfully mapped: {len(unmapped_states):,} rows")
print(unmapped_states[['state_raw', 'state_canonical']].drop_duplicates().head(10))

print("\\nStates that needed manual review (same as raw):")
manual_review = raw_election_results_1951_2019[
    raw_election_results_1951_2019['state_raw'] == raw_election_results_1951_2019['state_canonical']
]['state_raw'].value_counts().head(10)
print(manual_review)

# ─── 4. Final save  ──────────────────────────
output_path = 'FilteredData/clean_election_results_1951_2019.csv'
raw_election_results_1951_2019.to_csv(output_path, index=False)
print(f"\\nFinal cleaned file saved: {output_path}")

# ─── 5. Quick data quality check ─────────────────────────────────────────
print("\\nData quality summary:")
print(f"  Total rows: {len(raw_election_results_1951_2019):,}")
print(f"  Years covered: {raw_election_results_1951_2019['election_year_clean'].min()}–{raw_election_results_1951_2019['election_year_clean'].max()}")
print(f"  Unique states: {raw_election_results_1951_2019['state_canonical'].nunique()}")
print(f"  Unique parties: {raw_election_results_1951_2019['party'].nunique()}")
print(f"  Winner records: {(raw_election_results_1951_2019['result'] == 'winner').sum()}")
print(f"  Vote share range: {raw_election_results_1951_2019['vote_share_pct'].min():.1f}% – {raw_election_results_1951_2019['vote_share_pct'].max():.1f}%")

Vote share stats (should look good):
count    89811.000000
mean         9.084370
std         16.207498
min          0.000000
25%          0.140000
50%          0.510000
75%          9.080000
max         97.690000
Name: vote_share_pct, dtype: float64
\nYear extraction verification:
election_year_clean
1951    1874
1957    1595
1962    1987
1967    2324
1971    2782
1977    2418
1980    4293
1984    5490
1989    6061
1991    8461
Name: count, dtype: Int64
\nSample year mapping:
          election_year  election_year_clean
0  1951 (1st LOK SABHA)                 1951
1  1951 (1st LOK SABHA)                 1951
2  1951 (1st LOK SABHA)                 1951
3  1951 (1st LOK SABHA)                 1951
4  1951 (1st LOK SABHA)                 1951
5  1951 (1st LOK SABHA)                 1951
6  1951 (1st LOK SABHA)                 1951
7  1951 (1st LOK SABHA)                 1951
8  1951 (1st LOK SABHA)                 1951
9  1951 (1st LOK SABHA)                 1951
\nState mapping verifica

### Table 2: raw_loksabha_1962_2019

In [ ]:
raw_loksabha_1962_2019.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8047 entries, 0 to 8046
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Pc_name         8047 non-null   object 
 1   no              8047 non-null   object 
 2   type            8015 non-null   object 
 3   state           8047 non-null   object 
 4   candidate_name  8047 non-null   object 
 5   party           8047 non-null   object 
 6   electors        8047 non-null   object 
 7   votes           8047 non-null   object 
 8   Turnout         8033 non-null   object 
 9   margin          8047 non-null   object 
 10  margin%         8032 non-null   object 
 11  year            8046 non-null   float64
dtypes: float64(1), object(11)
memory usage: 754.5+ KB


In [30]:
raw_loksabha_1962_2019.head()

,Pc_name,no,type,state,candidate_name,party,electors,votes,Turnout,margin,margin%,year
0,Adilabad,36,GEN,Andhra Pradesh,G. Narayan Reddy,Indian National Congress,"4,04,283","2,20,383",54.50%,"89,085",40.40%,1962
1,Adoni,27,GEN,Andhra Pradesh,Pendekanti Venkatasubbaiah,Indian National Congress,"4,19,077","2,52,379",60.20%,"33,022",13.10%,1962
2,Agra,433,GEN,Uttar Pradesh [1947 - 1999],Seth Achal Singh,Indian National Congress,"4,33,164","2,75,663",63.60%,"54,351",19.70%,1962
3,Ahmedabad,120,GEN,Gujarat,Indulal Kanaiyalal Yagnik,Nutan Maha Gujarat Janta Parisha,"4,33,392","2,70,346",62.40%,"21,592",8.00%,1962
4,Ahmednagar,245,GEN,Maharashtra,Motilal Kundanmal Firodya,Indian National Congress,"4,03,913","2,22,091",55.00%,"14,038",6.30%,1962


In [ ]:
# ─── DEFINE STATE MAPPING ───────────────────────────────
state_mapping = dict(zip(
    state_name_variants['variant'], 
    state_name_variants['canonical_name']
))

print(f"State mapping created: {len(state_mapping)} entries")
print("Sample mappings:")
for i, (k, v) in enumerate(list(state_mapping.items())[:10]):
    print(f"  {k:<25} → {v}")

State mapping created: 78 entries
Sample mappings:
  J&K                       → Jammu and Kashmir
  J & K                     → Jammu and Kashmir
  JandK                     → Jammu and Kashmir
  Jammu and Kashmir         → Jammu and Kashmir
  Jammu & Kashmir           → Jammu and Kashmir
  Jammu And Kashmir         → Jammu and Kashmir
  JAMMU AND KASHMIR         → Jammu and Kashmir
  jammu and kashmir         → Jammu and Kashmir
  J.&K.                     → Jammu and Kashmir
  Jammu-Kashmir             → Jammu and Kashmir


In [ ]:
# ────────────────────────────────
print("Electors column before final clean:")
print("dtype:", raw_loksabha_1962_2019['electors'].dtype)
print("Unique values sample:")
print(raw_loksabha_1962_2019['electors'].dropna().unique()[:20])

# ─── CONVERT ALL POSSIBLE NUMBERIC DATA TO INT AND FLOAT──────────────────────────────────────────
def safe_numeric_convert(series):
    """Convert ANYTHING to numeric, handling every possible edge case."""
    return pd.to_numeric(
        series.astype(str)
        .str.replace(r'[^0-9.,\-]', '', regex=True)  # Keep ONLY numbers + . , -
        .str.replace(',', '', regex=False)
        .replace('', np.nan)
        .replace('nan', np.nan),
        errors='coerce'
    )

# Apply to ALL numeric columns
numeric_cols = ['electors', 'votes', 'Turnout', 'margin', 'margin%']
for col in numeric_cols:
    if col in raw_loksabha_1962_2019.columns:
        raw_loksabha_1962_2019[col] = safe_numeric_convert(raw_loksabha_1962_2019[col])
        print(f" {col}: {raw_loksabha_1962_2019[col].dtype} ({raw_loksabha_1962_2019[col].notna().sum():,} valid)")

# ─── Now verification will work ─────────────────────────────────────────
print("\\nNumeric ranges:")
for col in ['electors', 'votes', 'Turnout']:
    print(f"  {col}: {raw_loksabha_1962_2019[col].min():.0f} – {raw_loksabha_1962_2019[col].max():.0f}")

# ─── State standardization (if not done) ────────────────────────────────
if 'state_raw' not in raw_loksabha_1962_2019.columns:
    raw_loksabha_1962_2019['state_raw'] = raw_loksabha_1962_2019['state']
    raw_loksabha_1962_2019['state_canonical'] = raw_loksabha_1962_2019['state'].map(state_mapping).fillna(raw_loksabha_1962_2019['state'])

raw_loksabha_1962_2019['election_year_clean'] = raw_loksabha_1962_2019['year'].astype('Int64')

# ─── SAVE ───────────────────────────────────────────────────────────────
raw_loksabha_1962_2019.to_csv('FilteredData/clean_loksabha_1962_2019.csv', index=False)
print("\\nSAVED: clean_loksabha_1962_2019.csv")
print("Shape:", raw_loksabha_1962_2019.shape)

Electors column before final clean:
dtype: float64
Unique values sample:
[404283. 419077. 433164. 433392. 403913. 432778. 430435. 450141. 466414.
 446090. 401359. 451563. 478247. 439952. 445802. 475329. 418212. 457869.
 423038. 427862.]
 electors: float64 (8,007 valid)
 votes: float64 (7,993 valid)
 Turnout: float64 (7,993 valid)
 margin: float64 (7,993 valid)
 margin%: float64 (7,993 valid)
\nNumeric ranges:
  electors: 1 – 3368399
  votes: 11897 – 1763757
  Turnout: 5 – 108
\nSAVED: clean_loksabha_1962_2019.csv
Shape: (8047, 15)


### Table 3: clean_election_results_2024.csv

In [ ]:
raw_election_results_2024.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8902 entries, 0 to 8901
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   _id           8902 non-null   int64 
 1   State         8902 non-null   object
 2   PC No         8902 non-null   int64 
 3   PC Name       8902 non-null   object
 4   Sl no         8902 non-null   int64 
 5   Candidate     8902 non-null   object
 6   Party         8902 non-null   object
 7   EVM Votes     8902 non-null   object
 8   Postal Votes  8902 non-null   object
 9   Total Votes   8902 non-null   object
 10  Vote Share    8902 non-null   object
dtypes: int64(3), object(8)
memory usage: 765.1+ KB


In [ ]:
raw_election_results_2024.head()

,_id,State,PC No,PC Name,Sl no,Candidate,Party,EVM Votes,Postal Votes,Total Votes,Vote Share
0,1,Andhra Pradesh,1,Araku (ST),1,GUMMA THANUJA RANI,Yuvajana Sramika Rythu Congress Party,471470,5535,477005,40.96
1,2,Andhra Pradesh,1,Araku (ST),2,KOTHAPALLI GEETHA,Bharatiya Janata Party,417113,9312,426425,36.62
2,3,Andhra Pradesh,1,Araku (ST),3,APPALANARASA PACHIPENTA,Communist Party of India (Marxist),119016,4113,123129,10.57
3,4,Andhra Pradesh,1,Araku (ST),4,AVASHYA LAHARI . VARAM,Bahujan Samaj Party,24858,892,25750,2.21
4,5,Andhra Pradesh,1,Araku (ST),5,SAMAREDDY BALAKRISHNA,Independent,9493,42,9535,0.82


In [ ]:
# ─── 3.1 Convert vote columns (already clean format) ─────────────────────
vote_cols = ['EVM Votes', 'Postal Votes', 'Total Votes']
for col in vote_cols:
    raw_election_results_2024[col] = pd.to_numeric(
        raw_election_results_2024[col], errors='coerce'
    )
    print(f"{col}: {raw_election_results_2024[col].dtype}")

# ─── 3.2 Convert Vote Share % to numeric ─────────────────────────────────
raw_election_results_2024['vote_share_pct'] = pd.to_numeric(
    raw_election_results_2024['Vote Share'], errors='coerce'
)

# ─── 3.3 State standardization ───────────────────────────────────────────
raw_election_results_2024['state_raw'] = raw_election_results_2024['State']
raw_election_results_2024['state_canonical'] = (
    raw_election_results_2024['State']
    .map(state_mapping)
    .fillna(raw_election_results_2024['State'])
)

# ─── 3.4 Add election year ───────────────────────────────────────────────
raw_election_results_2024['election_year_clean'] = 2024

# ─── 3.5 Add rank/position (Sl no) ───────────────────────────────────────
raw_election_results_2024['rank_clean'] = raw_election_results_2024['Sl no']

# ─── 3.6 Verification ───────────────────────────────────────────────────
print("\nTable 3 verification:")
print("Vote columns:")
for col in vote_cols:
    print(f"  {col}: {raw_election_results_2024[col].min():,.0f} – {raw_election_results_2024[col].max():,.0f}")

print(f"Vote share: {raw_election_results_2024['vote_share_pct'].min():.1f}% – {raw_election_results_2024['vote_share_pct'].max():.1f}%")
print(f"States: {raw_election_results_2024['state_canonical'].nunique()} unique")

# ─── 3.7 Save ───────────────────────────────────────────────────────────
output_path = 'FilteredData/clean_election_results_2024.csv'
raw_election_results_2024.to_csv(output_path, index=False)
print(f"\nSAVED: {output_path}")
print(f"Shape: {raw_election_results_2024.shape}")

EVM Votes: float64
Postal Votes: float64
Total Votes: float64

Table 3 verification:
Vote columns:
  EVM Votes: 58 – 1,468,549
  Postal Votes: 1 – 19,827
  Total Votes: 61 – 1,471,885
Vote share: 0.0% – 78.5%
States: 36 unique

SAVED: FilteredData/clean_election_results_2024.csv
Shape: (8902, 16)


In [ ]:
# tomorrow clean these tables:
# raw_ref_party_master	Party names, abbreviations, and party type are inconsistent	Create canonical party master and abbreviation map. 
# raw_constituency_summary	Hierarchical categories only; totals are incomplete in some rows	Treat as metadata/supporting table, not a core analytical table. 
# raw_literacy_1951_2011	Small table, but clean already	Can be used directly after minor renaming. 
# raw_parliament_1951_2014	Column names are messy, state names are in uppercase, age/category sparse	Rename columns, normalize state names, clean missingness, convert votes and electors. 
# raw_state_sdp	Values are object strings, missing later years	Convert annual columns to numeric, reshape to long format. 
# raw_state_gdp_share	gdpshare missing in some rows; state names include combined historical entries	Clean numeric share and standardize states with historical split logic. 

### Table 4: raw_ref_party_master

In [4]:
raw_ref_party_master.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10153 entries, 0 to 10152
Data columns (total 21 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Assembly                 10153 non-null  object 
 1   State_Name               10153 non-null  object 
 2   Party_Name               10147 non-null  object 
 3   Party_Type               7113 non-null   object 
 4   Party_ID                 10147 non-null  float64
 5   Frequent_Abbreviation    10088 non-null  object 
 6   Last_Abbreviation        10088 non-null  object 
 7   Abbreviations            10084 non-null  object 
 8   Start_Year               10152 non-null  float64
 9   Last_Year                10152 non-null  float64
 10  No_Assemblies_Contested  10153 non-null  int64  
 11  Assemblies_Contested     10153 non-null  object 
 12  Candidates_Contested     10153 non-null  int64  
 13  Candidates_Represented   10153 non-null  int64  
 14  Females_Contested     

In [5]:
raw_ref_party_master.head()

,Assembly,State_Name,Party_Name,Party_Type,Party_ID,Frequent_Abbreviation,Last_Abbreviation,Abbreviations,Start_Year,Last_Year,...,Assemblies_Contested,Candidates_Contested,Candidates_Represented,Females_Contested,Females_Represented,SC_Seats_Contested,SC_Seats_Represented,ST_Seats_Contested,ST_Seats_Represented,BiPoll_Contested
0,Lok_Sabha,All_States,CONGRESS,National Party,3482.0,INC,INC,INC|INC(I),1962.0,2019.0,...,3|4|5|6|7|8|9|10|11|12|13|14|15|16|17,7392,3307,676,277,1091,440,585,293,235
1,Lok_Sabha,All_States,Bharatiya Janata Party,National Party,1605.0,BJP,BJP,BJP,1981.0,2019.0,...,7|8|9|10|11|12|13|14|15|16|17,3904,1601,309,161,548,228,349,144,114
2,Lok_Sabha,All_States,COMMUNIST MARXIST PARTY OF INDIA,State-based Party,14635.0,CPM,CPIM,CPM|CPI(M)|CPI (M)|CPI(Marxist),1965.0,2019.0,...,3|4|5|6|7|8|9|10|11|12|13|14|15|16|17,1003,370,78,30,154,69,76,21,20
3,Lok_Sabha,All_States,Bharatiya Lok Dal,National Party,1711.0,BLD,BLD,BLD,1977.0,1977.0,...,6,405,295,13,8,56,45,30,20,0
4,Lok_Sabha,All_States,Janata Dal,National Party,4217.0,JD,JD,JD,1989.0,1998.0,...,9|10|11|12,949,255,40,9,143,49,83,9,6


In [8]:
raw_ref_party_master.columns

Index(['Assembly', 'State_Name', 'Party_Name', 'Party_Type', 'Party_ID',
       'Frequent_Abbreviation', 'Last_Abbreviation', 'Abbreviations',
       'Start_Year', 'Last_Year', 'No_Assemblies_Contested',
       'Assemblies_Contested', 'Candidates_Contested',
       'Candidates_Represented', 'Females_Contested', 'Females_Represented',
       'SC_Seats_Contested', 'SC_Seats_Represented', 'ST_Seats_Contested',
       'ST_Seats_Represented', 'BiPoll_Contested', 'party_canonical',
       'party_abbr_clean', 'state_raw'],
      dtype='object')

In [6]:
# ─── 1. Numeric conversion (already mostly done) ─────────────────────────
numeric_cols = ['Party_ID', 'Start_Year', 'Last_Year', 'Candidates_Contested', 
                'Candidates_Represented', 'Females_Contested', 'Females_Represented',
                'SC_Seats_Contested', 'SC_Seats_Represented', 'ST_Seats_Contested', 
                'ST_Seats_Represented', 'BiPoll_Contested']

for col in numeric_cols:
    if col in raw_ref_party_master.columns:
        raw_ref_party_master[col] = pd.to_numeric(raw_ref_party_master[col], errors='coerce')

In [9]:
state_mapping = dict(zip(
    state_name_variants['variant'],
    state_name_variants['canonical_name']
))

print(f"state_mapping ready: {len(state_mapping)} entries")
print(list(state_mapping.items())[:10])

state_mapping ready: 78 entries
[('J&K', 'Jammu and Kashmir'), ('J & K', 'Jammu and Kashmir'), ('JandK', 'Jammu and Kashmir'), ('Jammu and Kashmir', 'Jammu and Kashmir'), ('Jammu & Kashmir', 'Jammu and Kashmir'), ('Jammu And Kashmir', 'Jammu and Kashmir'), ('JAMMU AND KASHMIR', 'Jammu and Kashmir'), ('jammu and kashmir', 'Jammu and Kashmir'), ('J.&K.', 'Jammu and Kashmir'), ('Jammu-Kashmir', 'Jammu and Kashmir')]


In [ ]:
# ─── 2. Create canonical party name and abbreviation ─────────────────────
# Take Frequent_Abbreviation as primary, fallback to Party_Name
raw_ref_party_master['party_canonical'] = raw_ref_party_master['Frequent_Abbreviation'].fillna(raw_ref_party_master['Party_Name'])
raw_ref_party_master['party_abbr_clean'] = raw_ref_party_master['Frequent_Abbreviation'].fillna('')

# ─── 3. State standardization ───────────────────────────────────────────
raw_ref_party_master['state_raw'] = raw_ref_party_master['State_Name']
raw_ref_party_master['state_canonical'] = (
    raw_ref_party_master['State_Name']
    .map(state_mapping)
    .fillna(raw_ref_party_master['State_Name'])
)

In [11]:
raw_ref_party_master['party_type_clean'] = raw_ref_party_master['Party_Type'].fillna('Unknown')

raw_ref_party_master['active_1952_2024'] = (
    (raw_ref_party_master['Start_Year'] <= 2024) &
    (raw_ref_party_master['Last_Year'] >= 1952)
)

In [12]:
raw_ref_party_master.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10153 entries, 0 to 10152
Data columns (total 27 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Assembly                 10153 non-null  object 
 1   State_Name               10153 non-null  object 
 2   Party_Name               10147 non-null  object 
 3   Party_Type               7113 non-null   object 
 4   Party_ID                 10147 non-null  float64
 5   Frequent_Abbreviation    10088 non-null  object 
 6   Last_Abbreviation        10088 non-null  object 
 7   Abbreviations            10084 non-null  object 
 8   Start_Year               10152 non-null  float64
 9   Last_Year                10152 non-null  float64
 10  No_Assemblies_Contested  10153 non-null  int64  
 11  Assemblies_Contested     10153 non-null  object 
 12  Candidates_Contested     10153 non-null  int64  
 13  Candidates_Represented   10153 non-null  int64  
 14  Females_Contested     

In [13]:
raw_ref_party_master.head()

,Assembly,State_Name,Party_Name,Party_Type,Party_ID,Frequent_Abbreviation,Last_Abbreviation,Abbreviations,Start_Year,Last_Year,...,SC_Seats_Represented,ST_Seats_Contested,ST_Seats_Represented,BiPoll_Contested,party_canonical,party_abbr_clean,state_raw,state_canonical,party_type_clean,active_1952_2024
0,Lok_Sabha,All_States,CONGRESS,National Party,3482.0,INC,INC,INC|INC(I),1962.0,2019.0,...,440,585,293,235,INC,INC,All_States,All_States,National Party,True
1,Lok_Sabha,All_States,Bharatiya Janata Party,National Party,1605.0,BJP,BJP,BJP,1981.0,2019.0,...,228,349,144,114,BJP,BJP,All_States,All_States,National Party,True
2,Lok_Sabha,All_States,COMMUNIST MARXIST PARTY OF INDIA,State-based Party,14635.0,CPM,CPIM,CPM|CPI(M)|CPI (M)|CPI(Marxist),1965.0,2019.0,...,69,76,21,20,CPM,CPM,All_States,All_States,State-based Party,True
3,Lok_Sabha,All_States,Bharatiya Lok Dal,National Party,1711.0,BLD,BLD,BLD,1977.0,1977.0,...,45,30,20,0,BLD,BLD,All_States,All_States,National Party,True
4,Lok_Sabha,All_States,Janata Dal,National Party,4217.0,JD,JD,JD,1989.0,1998.0,...,49,83,9,6,JD,JD,All_States,All_States,National Party,True


In [15]:
#Exporting the dataset:
raw_ref_party_master.to_csv('FilteredData/clean_ref_party_master.csv', index=False)
print("Saved FilteredData/clean_ref_party_master.csv")

Saved FilteredData/clean_ref_party_master.csv


---

In [16]:
raw_constituency_summary.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21680 entries, 0 to 21679
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   State/UT & Code           21680 non-null  object 
 1   Constituency Name & Code  21680 non-null  object 
 2   Code Name                 21680 non-null  object 
 3   Category                  21680 non-null  object 
 4   Men                       5962 non-null   float64
 5   Women                     5962 non-null   float64
 6   Third Gender              5962 non-null   float64
 7   Total                     16802 non-null  float64
dtypes: float64(4), object(4)
memory usage: 1.3+ MB


In [17]:
raw_constituency_summary.head()

,State/UT & Code,Constituency Name & Code,Code Name,Category,Men,Women,Third Gender,Total
0,Andhra Pradesh-S01,Araku-ST,S01_1,Candidates - Nominated,18.0,5.0,0.0,23.0
1,Andhra Pradesh-S01,Araku-ST,S01_1,Candidates - Nomination Rejected,7.0,1.0,0.0,8.0
2,Andhra Pradesh-S01,Araku-ST,S01_1,Candidates - Withdrawn,2.0,0.0,0.0,2.0
3,Andhra Pradesh-S01,Araku-ST,S01_1,Candidates - Contested,9.0,4.0,0.0,13.0
4,Andhra Pradesh-S01,Araku-ST,S01_1,Candidates - Forfeited Deposit,9.0,2.0,0.0,11.0


### Table 5: raw_constituency_summary

In [18]:
# backup bro
constituency_summary = raw_constituency_summary.copy()

In [19]:
# Split State/UT & Code into state name and code
constituency_summary[['state_name_raw', 'state_code']] = (
    constituency_summary['State/UT & Code']
    .astype(str)
    .str.rsplit('-', n=1, expand=True)
)

# Normalize state name using state variant map if available
if 'state_name_variants' in globals():
    state_mapping = dict(zip(
        state_name_variants['variant'],
        state_name_variants['canonical_name']
    ))
    constituency_summary['state_canonical'] = (
        constituency_summary['state_name_raw']
        .map(state_mapping)
        .fillna(constituency_summary['state_name_raw'])
    )
else:
    constituency_summary['state_canonical'] = constituency_summary['state_name_raw']

In [20]:
# Split constituency name and code hint
constituency_summary[['constituency_name', 'constituency_type']] = (
    constituency_summary['Constituency Name & Code']
    .astype(str)
    .str.rsplit('-', n=1, expand=True)
)

In [21]:
# Clean category labels
constituency_summary['category_clean'] = (
    constituency_summary['Category']
    .astype(str)
    .str.strip()
)

In [22]:
# Recompute total where possible
constituency_summary['total_recomputed'] = (
    constituency_summary[['Men', 'Women', 'Third Gender']]
    .sum(axis=1, skipna=True)
)

# Flag rows where total is missing or differs
constituency_summary['total_missing'] = constituency_summary['Total'].isna()
constituency_summary['total_diff'] = (
    constituency_summary['Total'] - constituency_summary['total_recomputed']
)

In [23]:
constituency_summary.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21680 entries, 0 to 21679
Data columns (total 17 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   State/UT & Code           21680 non-null  object 
 1   Constituency Name & Code  21680 non-null  object 
 2   Code Name                 21680 non-null  object 
 3   Category                  21680 non-null  object 
 4   Men                       5962 non-null   float64
 5   Women                     5962 non-null   float64
 6   Third Gender              5962 non-null   float64
 7   Total                     16802 non-null  float64
 8   state_name_raw            21680 non-null  object 
 9   state_code                21680 non-null  object 
 10  state_canonical           21680 non-null  object 
 11  constituency_name         21680 non-null  object 
 12  constituency_type         21680 non-null  object 
 13  category_clean            21680 non-null  object 
 14  total_

In [24]:
constituency_summary.head()

,State/UT & Code,Constituency Name & Code,Code Name,Category,Men,Women,Third Gender,Total,state_name_raw,state_code,state_canonical,constituency_name,constituency_type,category_clean,total_recomputed,total_missing,total_diff
0,Andhra Pradesh-S01,Araku-ST,S01_1,Candidates - Nominated,18.0,5.0,0.0,23.0,Andhra Pradesh,S01,Andhra Pradesh,Araku,ST,Candidates - Nominated,23.0,False,0.0
1,Andhra Pradesh-S01,Araku-ST,S01_1,Candidates - Nomination Rejected,7.0,1.0,0.0,8.0,Andhra Pradesh,S01,Andhra Pradesh,Araku,ST,Candidates - Nomination Rejected,8.0,False,0.0
2,Andhra Pradesh-S01,Araku-ST,S01_1,Candidates - Withdrawn,2.0,0.0,0.0,2.0,Andhra Pradesh,S01,Andhra Pradesh,Araku,ST,Candidates - Withdrawn,2.0,False,0.0
3,Andhra Pradesh-S01,Araku-ST,S01_1,Candidates - Contested,9.0,4.0,0.0,13.0,Andhra Pradesh,S01,Andhra Pradesh,Araku,ST,Candidates - Contested,13.0,False,0.0
4,Andhra Pradesh-S01,Araku-ST,S01_1,Candidates - Forfeited Deposit,9.0,2.0,0.0,11.0,Andhra Pradesh,S01,Andhra Pradesh,Araku,ST,Candidates - Forfeited Deposit,11.0,False,0.0


In [28]:
# Export
constituency_summary.to_csv('FilteredData/clean_constituency_summary.csv', index=False)

print("Saved FilteredData/clean_constituency_summary.csv")
print(constituency_summary[['State/UT & Code', 'state_name_raw', 'state_code', 'Constituency Name & Code', 'constituency_name', 'constituency_type']].head())
print(constituency_summary['Category'].value_counts().head(10))

Saved FilteredData/clean_constituency_summary.csv
      State/UT & Code  state_name_raw state_code Constituency Name & Code  \
0  Andhra Pradesh-S01  Andhra Pradesh        S01                 Araku-ST   
1  Andhra Pradesh-S01  Andhra Pradesh        S01                 Araku-ST   
2  Andhra Pradesh-S01  Andhra Pradesh        S01                 Araku-ST   
3  Andhra Pradesh-S01  Andhra Pradesh        S01                 Araku-ST   
4  Andhra Pradesh-S01  Andhra Pradesh        S01                 Araku-ST   

  constituency_name constituency_type  
0             Araku                ST  
1             Araku                ST  
2             Araku                ST  
3             Araku                ST  
4             Araku                ST  
Category
Candidates - Nominated              542
Candidates - Nomination Rejected    542
Candidates - Withdrawn              542
Candidates - Contested              542
Candidates - Forfeited Deposit      542
Electors - General                  54

---

### Table 6: raw_literacy_1951_2011 

In [25]:
raw_literacy_1951_2011.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Year     7 non-null      int64  
 1   Persons  7 non-null      float64
 2   Males    7 non-null      float64
 3   Females  7 non-null      float64
dtypes: float64(3), int64(1)
memory usage: 356.0 bytes


In [26]:
raw_literacy_1951_2011.head()

,Year,Persons,Males,Females
0,1951,18.33,27.16,8.86
1,1961,28.30,40.40,15.35
2,1971,34.45,45.96,21.97
3,1981,43.57,56.38,29.76
4,1991,52.21,64.13,39.29


In [27]:
# ─── Rename for consistency with election tables ─────────────────────────
raw_literacy_1951_2011_clean = raw_literacy_1951_2011.rename(columns={
    'Year': 'election_year_clean',
    'Persons': 'literacy_persons_pct',
    'Males': 'literacy_males_pct', 
    'Females': 'literacy_females_pct'
}).copy()


In [29]:
# Add India identifier for joining later
raw_literacy_1951_2011_clean['state_canonical'] = 'India'

# Save
output_path = 'FilteredData/clean_literacy_1951_2011.csv'
raw_literacy_1951_2011_clean.to_csv(output_path, index=False)

print("Saved:", output_path)
print("\\nLiteracy trends ready:")
print(raw_literacy_1951_2011_clean[['election_year_clean', 'literacy_persons_pct']].round(1).to_string(index=False))

Saved: FilteredData/clean_literacy_1951_2011.csv
\nLiteracy trends ready:
 election_year_clean  literacy_persons_pct
                1951                  18.3
                1961                  28.3
                1971                  34.4
                1981                  43.6
                1991                  52.2
                2001                  64.8
                2011                  73.0


---

### Table 7: raw_parliament_1951_2014

In [31]:
raw_parliament_1951_2014.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74930 entries, 0 to 74929
Data columns (total 11 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   YEAR      74930 non-null  int64  
 1   STATE     74930 non-null  object 
 2   PC        74930 non-null  object 
 3   NAME      74930 non-null  object 
 4   SEX       71462 non-null  object 
 5   PARTY     74930 non-null  object 
 6   AGE       13505 non-null  float64
 7   CATEGORY  13505 non-null  object 
 8   VOTES     74930 non-null  int64  
 9   ELECTORS  69495 non-null  float64
 10  #         74930 non-null  int64  
dtypes: float64(2), int64(3), object(6)
memory usage: 6.3+ MB


In [32]:
raw_parliament_1951_2014.head()

,YEAR,STATE,PC,NAME,SEX,PARTY,AGE,CATEGORY,VOTES,ELECTORS,#
0,1951,AJMER,AJMER NORTH,JAWALA PRASHAD,NaN,INC,NaN,NaN,46679,162327.0,1
1,1951,AJMER,AJMER NORTH,CHAND KARAN,NaN,BJS,NaN,NaN,28990,162327.0,2
2,1951,AJMER,AJMER NORTH,DINO MAL,NaN,PURP,NaN,NaN,10778,162327.0,3
3,1951,AJMER,AJMER NORTH,BAJORIA BADRIDAS,NaN,IND,NaN,NaN,6153,162327.0,4
4,1951,AJMER,AJMER NORTH,RANG RAJ MEHTA,NaN,IND,NaN,NaN,4565,162327.0,5


In [ ]:
# 1. _______kaam ki chij____________
raw_parliament_1951_2014_clean = raw_parliament_1951_2014.rename(columns={
    'YEAR': 'election_year_clean',
    'STATE': 'state_raw',
    'PC': 'constituency',
    'NAME': 'candidate',
    'SEX': 'gender',
    'PARTY': 'party',
    'AGE': 'age',
    'CATEGORY': 'category',
    'VOTES': 'votes_received',
    'ELECTORS': 'electors',
    '#': 'rank'
}).copy()

In [ ]:
# ─── 2. Numeric conversion (already mostly done) ─────────────────────────
numeric_cols = ['votes_received', 'electors', 'age', 'rank']
for col in numeric_cols:
    raw_parliament_1951_2014_clean[col] = pd.to_numeric(
        raw_parliament_1951_2014_clean[col], errors='coerce'
    )

In [35]:
# ─── 3. State standardization ───────────────────────────────────────────
raw_parliament_1951_2014_clean['state_raw'] = raw_parliament_1951_2014_clean['state_raw'].str.upper()
raw_parliament_1951_2014_clean['state_canonical'] = (
    raw_parliament_1951_2014_clean['state_raw']
    .map(state_mapping)
    .fillna(raw_parliament_1951_2014_clean['state_raw'])
)

In [36]:
# ─── 4. Compute vote share ───────────────────────────────────────────────
raw_parliament_1951_2014_clean['vote_share_pct'] = (
    (raw_parliament_1951_2014_clean['votes_received'] / raw_parliament_1951_2014_clean['electors']) * 100
)

In [37]:
# ─── 5. Verification ────────────────────────────────────────────────────
print("Parliament data verification:-")
print(f"Rows: {len(raw_parliament_1951_2014_clean):,}")
print(f"Years: {raw_parliament_1951_2014_clean['election_year_clean'].min()}–{raw_parliament_1951_2014_clean['election_year_clean'].max()}")
print(f"States: {raw_parliament_1951_2014_clean['state_canonical'].nunique()}")

print("\\nSample data:")
print(raw_parliament_1951_2014_clean[['election_year_clean', 'state_canonical', 'constituency', 'candidate', 'party', 'votes_received', 'vote_share_pct']].head())

print("\\nTop vote shares:")
print(raw_parliament_1951_2014_clean.nlargest(5, 'vote_share_pct')[['candidate', 'state_canonical', 'vote_share_pct']].to_string(index=False))


Parliament data verification:-
Rows: 74,930
Years: 1951–2009
States: 50
\nSample data:
   election_year_clean state_canonical constituency         candidate party  \
0                 1951           AJMER  AJMER NORTH    JAWALA PRASHAD   INC   
1                 1951           AJMER  AJMER NORTH       CHAND KARAN   BJS   
2                 1951           AJMER  AJMER NORTH          DINO MAL  PURP   
3                 1951           AJMER  AJMER NORTH  BAJORIA BADRIDAS   IND   
4                 1951           AJMER  AJMER NORTH    RANG RAJ MEHTA   IND   

   votes_received  vote_share_pct  
0           46679       28.756153  
1           28990       17.859013  
2           10778        6.639684  
3            6153        3.790497  
4            4565        2.812225  
\nTop vote shares:
            candidate state_canonical  vote_share_pct
     RAM VILAS PASWAN           Bihar       68.271931
DIGVIJOY NARAIN SINGH           Bihar       64.259192
          LALU PRASAD           Bihar    

In [38]:
# ─── 6. Save ────────────────────────────────────────────────────────────
output_path = 'FilteredData/clean_parliament_1951_2014.csv'
raw_parliament_1951_2014_clean.to_csv(output_path, index=False)
print(f"\\n SAVED: {output_path}")

\n SAVED: FilteredData/clean_parliament_1951_2014.csv


### Table 8 - raw_state_gdp_share

In [40]:
raw_state_gdp_share.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 291 entries, 0 to 290
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   fiscal_year  291 non-null    object 
 1   state        291 non-null    object 
 2   state_size   291 non-null    object 
 3   gdp_share    261 non-null    float64
 4   unit         291 non-null    object 
 5   notes        40 non-null     object 
dtypes: float64(1), object(5)
memory usage: 13.8+ KB


In [41]:
raw_state_gdp_share.head()

,fiscal_year,state,state_size,gdp_share,unit,notes
0,2023-24,Andhra Pradesh,Large,9.7,gdp_share in percentage,gdp_share: Data for Andhra Pradesh and Telanga...
1,2023-24,Assam,Large,1.9,gdp_share in percentage,NaN
2,2023-24,Bihar,Large,4.3,gdp_share in percentage,gdp_share:Data for Bihar and Jharkhand Together
3,2023-24,Chhattisgarh,Large,1.7,gdp_share in percentage,NaN
4,2023-24,Delhi,Large,3.6,gdp_share in percentage,NaN


In [42]:
# ─── 1. Extract election year from fiscal year ───────────────────────────
raw_state_gdp_share['election_year_clean'] = (
    raw_state_gdp_share['fiscal_year']
    .str[:4]  # Take first 4 digits: "2023-24" → "2023"
    .astype(int)
)

# ─── 2. State standardization ───────────────────────────────────────────
raw_state_gdp_share['state_raw'] = raw_state_gdp_share['state']
raw_state_gdp_share['state_canonical'] = (
    raw_state_gdp_share['state']
    .map(state_mapping)
    .fillna(raw_state_gdp_share['state'])
)

# ─── 3. Historical split logic (from notes) ─────────────────────────────
# Bihar + Jharkhand before 2000
mask_bihar_combined = (
    (raw_state_gdp_share['state'] == 'Bihar') &
    (raw_state_gdp_share['election_year_clean'] < 2000) &
    raw_state_gdp_share['notes'].str.contains('Jharkhand', na=False)
)
raw_state_gdp_share.loc[mask_bihar_combined, 'state_canonical'] = 'Bihar-Jharkhand Combined'

# Madhya Pradesh + Chhattisgarh before 2000
mask_mp_combined = (
    (raw_state_gdp_share['state'] == 'Madhya Pradesh') &
    (raw_state_gdp_share['election_year_clean'] < 2000) &
    raw_state_gdp_share['notes'].str.contains('Chhattisgarh', na=False)
)
raw_state_gdp_share.loc[mask_mp_combined, 'state_canonical'] = 'MP-Chhattisgarh Combined'

# Andhra Pradesh + Telangana before 2014
mask_ap_combined = (
    (raw_state_gdp_share['state'] == 'Andhra Pradesh') &
    (raw_state_gdp_share['election_year_clean'] < 2014) &
    raw_state_gdp_share['notes'].str.contains('Telangana', na=False)
)
raw_state_gdp_share.loc[mask_ap_combined, 'state_canonical'] = 'AP-Telangana Combined'

# ─── 4. Clean GDP share ─────────────────────────────────────────────────
raw_state_gdp_share['gdp_share_pct'] = raw_state_gdp_share['gdp_share'] * 100  # Convert to %


In [43]:
# ─── 5. Verification ────────────────────────────────────────────────────
print("GDP Share verification:")
print(f"Years: {raw_state_gdp_share['election_year_clean'].min()}–{raw_state_gdp_share['election_year_clean'].max()}")
print(f"States: {raw_state_gdp_share['state_canonical'].nunique()}")
print(f"GDP share coverage: {raw_state_gdp_share['gdp_share'].notna().sum()}/{len(raw_state_gdp_share)}")

print("\\nRecent years sample:")
recent = raw_state_gdp_share[raw_state_gdp_share['election_year_clean'] >= 2019].head(10)
print(recent[['election_year_clean', 'state_canonical', 'gdp_share_pct', 'notes']].round(1).to_string(index=False))

GDP Share verification:
Years: 1960–2023
States: 36
GDP share coverage: 261/291
\nRecent years sample:
 election_year_clean             state_canonical  gdp_share_pct                                                     notes
                2023              Andhra Pradesh          970.0 gdp_share: Data for Andhra Pradesh and Telangana Together
                2023                       Assam          190.0                                                       NaN
                2023                       Bihar          430.0           gdp_share:Data for Bihar and Jharkhand Together
                2023                Chhattisgarh          170.0                                                       NaN
                2023                       Delhi          360.0                                                       NaN
                2023 Andaman and Nicobar Islands            4.0                            gdp_share: data is for 2022-23
                2023           Arunachal Pr

In [44]:
# 6. Save ────────────────────────────────────────────────────────────
output_path = 'FilteredData/clean_state_gdp_share.csv'
raw_state_gdp_share.to_csv(output_path, index=False)
print(f"\\nSAVED: {output_path}")

\nSAVED: FilteredData/clean_state_gdp_share.csv
